In [ ]:
import os
import ffmpeg
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
import opensmile
from pydub import AudioSegment
import io
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

In [ ]:
INPUT_DIR = r"D:\Thesis\data\MP"
OUTPUT_DIR = r"D:\Thesis\data\Output"
HF_TOKEN = ""
WHISPER_MODEL_NAME = "medium"
WHISPER_LANGUAGE = "de"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=HF_TOKEN
).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

whisper_model = WhisperModel(WHISPER_MODEL_NAME, compute_type="float16")

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals
)

In [ ]:

def filter_short_segments(diarization_result, min_duration=1.1):
    filtered = []
    for segment, _, speaker_label in diarization_result.itertracks(yield_label=True):
        duration = segment.end - segment.start
        if duration >= min_duration:
            filtered.append({
                "start": segment.start,
                "end": segment.end,
                "speaker": speaker_label
            })
    return filtered

def assign_speaker_roles(segments):
    if not segments:
        return []
    first_speaker = segments[0]['speaker']
    unique_speakers = list({seg["speaker"] for seg in segments})
    if len(unique_speakers) != 4:
        for seg in segments:
            seg["role"] = seg["speaker"]
        return segments
    other_speaker = [spk for spk in unique_speakers if spk != first_speaker][0]
    for seg in segments:
        seg["role"] = "Interviewer" if seg["speaker"] == first_speaker else ("Participant" if seg["speaker"] == other_speaker else "Other")
    return segments

def align_transcription_with_speakers(whisper_segments, speaker_segments):
    aligned = []

    for seg in speaker_segments:
        start = seg["start"]
        end = seg["end"]
        matched_texts = []
        matched_starts = []
        matched_ends = []

        for ws in whisper_segments:
            if (ws.start >= start and ws.start <= end) or \
               (ws.end >= start and ws.end <= end) or \
               (ws.start <= start and ws.end >= end):
                matched_texts.append(ws.text)
                matched_starts.append(ws.start)
                matched_ends.append(ws.end)

        real_start = min(matched_starts) if matched_starts else start
        real_end = max(matched_ends) if matched_ends else end

        aligned.append({
            "start": real_start,
            "end": real_end,
            "speaker": seg["speaker"],
            "role": seg.get("role", "Unknown"),
            "transcript": " ".join(matched_texts)
        })

    return pd.DataFrame(aligned)



In [ ]:
def process_videos(input_dir, output_dir):
    summary_rows = []

    for subdir, _, files in os.walk(input_dir):
        mp4_files = [f for f in files if f.endswith(".mp4")]

        for video in mp4_files:
            base_name = os.path.splitext(video)[0]
            rel_path = os.path.relpath(subdir, input_dir)
            output_subdir = os.path.join(output_dir, rel_path.lower())
            os.makedirs(output_subdir, exist_ok=True)

            aligned_csv = os.path.join(output_subdir, f"{base_name}_aligned_segments.csv")
            if os.path.exists(aligned_csv):
                print(f"Skipping already processed video: {base_name}")
                continue

            video_path = os.path.join(subdir, video)
            audio_wav = os.path.join(output_subdir, base_name + ".wav")

            if not os.path.exists(audio_wav):
                ffmpeg.input(video_path).output(audio_wav, ac=1, ar=16000).run(overwrite_output=True)

            segments, _ = whisper_model.transcribe(audio_wav, language=WHISPER_LANGUAGE)
            whisper_segments = list(segments)

            diarization_result = diarization_pipeline(audio_wav, max_speakers=4)
            speaker_segments = filter_short_segments(diarization_result)
            speaker_segments = assign_speaker_roles(speaker_segments)

            transcript_df = align_transcription_with_speakers(whisper_segments, speaker_segments)
            transcript_df.to_csv(aligned_csv, index=False)

            full_audio = AudioSegment.from_wav(audio_wav)
            features_list = []
            for idx, segment in enumerate(speaker_segments):
                start_ms = int(segment["start"] * 1000)
                end_ms = int(segment["end"] * 1000)
                audio_chunk = full_audio[start_ms:end_ms]
                
                temp_wav = os.path.join(output_subdir, f"segment_{idx}.wav")
                audio_chunk.export(temp_wav, format="wav")
                features = smile.process_file(temp_wav)
                os.remove(temp_wav)
                features["speaker"] = segment["speaker"]
                features["role"] = segment["role"]
                features["start"] = segment["start"]
                features["end"] = segment["end"]
                features_list.append(features)

            features_df = pd.concat(features_list)
            features_csv = os.path.join(output_subdir, f"{base_name}_audio_features.csv")
            features_df.to_csv(features_csv, index=False)

            summary_rows.append({
                "video": base_name,
                "num_segments": len(speaker_segments),
                "num_speakers": len(set([seg["speaker"] for seg in speaker_segments])),
                "avg_segment_duration": np.mean([seg["end"] - seg["start"] for seg in speaker_segments]),
                "F0_mean": features_df["F0_sma"].mean() if "F0_sma" in features_df else np.nan,
                "Loudness_mean": features_df["Loudness_sma"].mean() if "Loudness_sma" in features_df else np.nan,
                "MFCC1_std": features_df["mfcc1"].std() if "mfcc1" in features_df else np.nan
            })

    return pd.DataFrame(summary_rows)

In [ ]:

def generate_summary_visualization(summary_df):
    plt.figure(figsize=(12, 6))
    plt.bar(summary_df["video"], summary_df["num_segments"])
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Video File")
    plt.ylabel("Number of Segments")
    plt.title("Number of Speaker Segments per Video")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.boxplot([
        summary_df["F0_mean"].dropna(),
        summary_df["Loudness_mean"].dropna(),
        summary_df["MFCC1_std"].dropna()
    ], labels=["F0 Mean", "Loudness Mean", "MFCC1 Std"])
    plt.title("Distribution of Selected OpenSMILE Features Across Videos")
    plt.grid(True)
    plt.show()


In [ ]:
summary_df = process_videos(INPUT_DIR, OUTPUT_DIR)
generate_summary_visualization(summary_df)
summary_df

In [ ]:

def aggregate_opensmile_features_by_label(eaf_annotations, opensmile_df):
    import numpy as np
    import pandas as pd

    records = []

    for _, row in eaf_annotations.iterrows():
        label = row["tier"]
        start, end = row["start"], row["end"]

        relevant_segments = opensmile_df[
            (opensmile_df["start"] < end) & (opensmile_df["end"] > start)
        ]

        sap_segments = relevant_segments[
            relevant_segments["role"].isin(["Participant", "Unknown", "Other"])
        ]

        gap_segments = relevant_segments

        exclude_cols = ["speaker", "role", "start", "end"]
        feature_cols = [col for col in opensmile_df.columns if col not in exclude_cols]

        sap_features = sap_segments[feature_cols].mean().add_suffix("_SAP")

        gap_features = gap_segments[feature_cols].mean().add_suffix("_GAP")

        combined = pd.concat([sap_features, gap_features])
        combined["tier"] = label
        combined["start"] = start
        combined["end"] = end

        records.append(combined)

    return pd.DataFrame(records)


In [ ]:

from pympi.Elan import Eaf

def load_eaf_annotations(eaf_path):
    eaf = Eaf(eaf_path)
    annotations = []
    for tier in eaf.get_tier_names():
        for start, end, value in eaf.get_annotation_data_for_tier(tier):
            annotations.append({
                "tier": tier,
                "start": start / 1000,
                "end": end / 1000
            })
    return pd.DataFrame(annotations)


In [ ]:

import glob
import os

def aggregate_all_videos(input_dir, output_dir):
    results = []

    for subdir, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith("_640.mp4"):
                base = file.replace("_640.mp4", "")
                base2 = file.replace(".mp4", "")
                eaf_file = os.path.join(subdir, f"{base}.eaf")
                feature_csv = os.path.join(output_dir, os.path.relpath(subdir, input_dir).lower(), f"{base2}_audio_features.csv")
                print(feature_csv)

                if not os.path.exists(feature_csv) or not os.path.exists(eaf_file):
                    print(f"Missing files for {base}")
                    continue

                annotations_df = load_eaf_annotations(eaf_file)
                features_df = pd.read_csv(feature_csv)

                sap_gap_df = aggregate_opensmile_features_by_label(annotations_df, features_df)
                sap_gap_df["video"] = base
                results.append(sap_gap_df)

    return pd.concat(results, ignore_index=True)


In [ ]:

final_sap_gap_df = aggregate_all_videos(INPUT_DIR, OUTPUT_DIR)
final_sap_gap_df.to_csv("aggregated_sap_gap_features.csv", index=False)
final_sap_gap_df.head()
